In [33]:
import tensorflow as tf
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.models import load_model
import numpy as np

In [34]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32
PROJECT_ROOT = "../.."
DATA_DIR = PROJECT_ROOT + "/data/mushrooms"

SEED = 1337


In [35]:
train_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,   # 20% validation
    subset="training",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

validation_dataset = tf.keras.utils.image_dataset_from_directory(
    DATA_DIR,
    validation_split=0.2,
    subset="validation",
    seed=SEED,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical'
)

Found 80 files belonging to 4 classes.
Using 64 files for training.
Found 80 files belonging to 4 classes.
Using 16 files for validation.


In [36]:
NUM_CLASSES = len(train_dataset.class_names)

base_model = EfficientNetB0(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

x = base_model.output
x = GlobalAveragePooling2D()(x)
x = Dense(128, activation='relu')(x)
outputs = Dense(NUM_CLASSES, activation='softmax')(x)

model = Model(inputs=base_model.input, outputs=outputs)

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

In [37]:
history = model.fit(
    train_dataset,
    validation_data=validation_dataset,
    epochs=10
)

Epoch 1/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 3s 899ms/step - accuracy: 0.3906 - loss: 1.3737 - val_accuracy: 0.7500 - val_loss: 1.0582
Epoch 2/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 295ms/step - accuracy: 0.8906 - loss: 0.8690 - val_accuracy: 0.8125 - val_loss: 0.8125
Epoch 3/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 298ms/step - accuracy: 0.9375 - loss: 0.5562 - val_accuracy: 0.9375 - val_loss: 0.5313
Epoch 4/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 303ms/step - accuracy: 0.9844 - loss: 0.3535 - val_accuracy: 1.0000 - val_loss: 0.3439
Epoch 5/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 307ms/step - accuracy: 0.9844 - loss: 0.2305 - val_accuracy: 1.0000 - val_loss: 0.2527
Epoch 6/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 295ms/step - accuracy: 0.9844 - loss: 0.1376 - val_accuracy: 1.0000 - val_loss: 0.1875
Epoch 7/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 291ms/step - accuracy: 1.0000 - loss: 0.0936 - val_accuracy: 1.0000 - val_loss: 0.1248
Epoch 8/10
2/2 ━━━━━━━━━━━━━━━━━━━━ 1s 295ms/step - accuracy: 1.0000 - loss: 0.0650 - val_accuracy: 1.0000 - val_loss:

In [38]:
MODEL_PATH = "../.. /src/ai_misko_asistentas/models/mushroom_model.keras"
MODEL_PATH = MODEL_PATH.replace(" ", "")
model.save(MODEL_PATH)

print("Model saved to:", MODEL_PATH)


Model saved to: ../../src/ai_misko_asistentas/models/mushroom_model.keras


In [39]:
CLASSES_PATH = "../.. /src/ai_misko_asistentas/mushroom_classes.txt"
CLASSES_PATH = CLASSES_PATH.replace(" ", "")

with open(CLASSES_PATH, "w", encoding="utf-8") as f:
    f.write("\n".join(train_dataset.class_names))


In [40]:
model = load_model(MODEL_PATH)

with open(CLASSES_PATH, "r", encoding="utf-8") as f:
    class_names = f.read().splitlines()

In [42]:
new_image_path = DATA_DIR + '/boletus_edulis/6.jpeg'
IMG_SIZE = (224, 224)

img = tf.keras.utils.load_img(
    new_image_path, target_size=IMG_SIZE
)

img_array = tf.keras.utils.img_to_array(img)

img_batch = tf.expand_dims(img_array, 0)

predictions = model.predict(img_batch)

class_names = train_dataset.class_names

predicted_index = np.argmax(predictions[0])

predicted_class = class_names[predicted_index]

confidence = np.max(predictions[0]) * 100

print(f"Predicted Mushroom: {predicted_class}")
print(f"Confidence: {confidence:.2f}%")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step
Predicted Mushroom: boletus_edulis
Confidence: 98.36%
